In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)
DATA_PATH = '/Users/yash/algothon-26/man-imperial-algothon-2026/data/2024-12-31'
# VAL_PATH = '/Users/yash/algothon-26/man-imperial-algothon-2026/data/2025-02-28' 
prices = pd.read_csv(os.path.join(DATA_PATH, 'prices.csv'), parse_dates=['date']).sort_values('date').set_index('date')
signals = pd.read_csv(os.path.join(DATA_PATH, 'signals.csv'), parse_dates=['date']).sort_values('date').set_index('date')
volumes = pd.read_csv(os.path.join(DATA_PATH, 'volumes.csv'), parse_dates=['date']).sort_values('date').set_index('date')
cash_rate = pd.read_csv(os.path.join(DATA_PATH, 'cash_rate.csv'), parse_dates=['date']).sort_values('date').set_index('date')
instruments = [f'INSTRUMENT_{i}' for i in range(1, 11)]
N = len(instruments)
returns = prices[instruments].pct_change().dropna()
signal_cols = [c for c in signals.columns if 'trend' in c]
print(f'returns: {returns.shape}, {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'signals: {len(signal_cols)} columns')

returns: (2850, 10), 2017-01-04 to 2024-12-31
signals: 40 columns


In [ ]:
def build_features(ret_df, sig_df, vol_df, cash_df, instruments, forward_days=63):
    frames = []
    cash_cols = ['3mo', '6mo', '1yr', '2yr', '5yr', '10yr']
    cash_avail = [c for c in cash_cols if c in cash_df.columns]
    cash_reindexed = cash_df[cash_avail].reindex(ret_df.index).ffill()
    for inst in instruments:
        f = pd.DataFrame(index=ret_df.index)
        r = ret_df[inst]
        f['ret_1d'] = r.shift(1)
        f['ret_5d'] = r.shift(1).rolling(5).mean()
        f['ret_10d'] = r.shift(1).rolling(10).mean()
        f['ret_21d'] = r.shift(1).rolling(21).mean()
        f['ret_63d'] = r.shift(1).rolling(63).mean()
        f['ret_126d'] = r.shift(1).rolling(126).mean()
        f['ret_252d'] = r.shift(1).rolling(252).mean()
        f['vol_10d'] = r.shift(1).rolling(10).std()
        f['vol_21d'] = r.shift(1).rolling(21).std()
        f['vol_63d'] = r.shift(1).rolling(63).std()
        f['vol_ratio_10_63'] = f['vol_10d'] / (f['vol_63d'] + 1e-8)
        f['vol_ratio_21_63'] = f['vol_21d'] / (f['vol_63d'] + 1e-8)
        f['mom_21'] = (1 + r.shift(1)).rolling(21).apply(np.prod, raw=True) - 1
        f['mom_63'] = (1 + r.shift(1)).rolling(63).apply(np.prod, raw=True) - 1
        f['mom_126'] = (1 + r.shift(1)).rolling(126).apply(np.prod, raw=True) - 1
        f['mom_252'] = (1 + r.shift(1)).rolling(252).apply(np.prod, raw=True) - 1
        f['skew_21'] = r.shift(1).rolling(21).skew()
        f['skew_63'] = r.shift(1).rolling(63).skew()
        f['kurt_63'] = r.shift(1).rolling(63).kurt()
        for h in [4, 8, 16, 32]:
            col = f'{inst}_trend{h}'
            if col in sig_df.columns:
                f[f'trend{h}'] = sig_df[col].reindex(ret_df.index).shift(1)
        
        f['trend_avg'] = f[[f'trend{h}' for h in [4, 8, 16, 32]]].mean(axis=1)
        f['trend_std'] = f[[f'trend{h}' for h in [4, 8, 16, 32]]].std(axis=1)
        f['trend_short_long'] = f['trend4'] - f['trend32']
        
        vol_col = f'{inst}_vol'
        if vol_col in vol_df.columns:
            v = vol_df[vol_col].reindex(ret_df.index).shift(1)
            f['vol_rel_5'] = v / (v.rolling(5).mean() + 1)
            f['vol_rel_21'] = v / (v.rolling(21).mean() + 1)
            f['vol_rel_63'] = v / (v.rolling(63).mean() + 1)
        
        for cc in cash_avail:
            f[f'rate_{cc}'] = cash_reindexed[cc].shift(1)
        if '2yr' in cash_avail and '10yr' in cash_avail:
            f['yield_spread'] = (cash_reindexed['10yr'] - cash_reindexed['2yr']).shift(1)
        if '3mo' in cash_avail and '10yr' in cash_avail:
            f['term_spread'] = (cash_reindexed['10yr'] - cash_reindexed['3mo']).shift(1)
        
        f['target'] = r.rolling(forward_days).mean().shift(-forward_days)
        f['instrument'] = inst
        frames.append(f)
    
    out = pd.concat(frames)
    feat_cols = [c for c in out.columns if c not in ['target', 'instrument']]
    return out, feat_cols

FORWARD = 63
feat_all, feat_cols = build_features(returns, signals, volumes, cash_rate, instruments, FORWARD)
print(f'features: {len(feat_cols)}')
print(feat_cols)

Features: 37
['ret_1d', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d', 'ret_126d', 'ret_252d', 'vol_10d', 'vol_21d', 'vol_63d', 'vol_ratio_10_63', 'vol_ratio_21_63', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'skew_21', 'skew_63', 'kurt_63', 'trend4', 'trend8', 'trend16', 'trend32', 'trend_avg', 'trend_std', 'trend_short_long', 'vol_rel_5', 'vol_rel_21', 'vol_rel_63', 'rate_3mo', 'rate_6mo', 'rate_1yr', 'rate_2yr', 'rate_5yr', 'rate_10yr', 'yield_spread', 'term_spread']


In [3]:
def optimize_mvo(mu, cov, max_weight=0.40):
    n = len(mu)
    w0 = np.ones(n) / n
    bounds = [(0.0, max_weight)] * n
    cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
    def obj(w):
        pv = np.sqrt(w @ cov @ w)
        return -(w @ mu) / pv if pv > 1e-10 else 0
    res = minimize(obj, w0, method='SLSQP', bounds=bounds, constraints=cons)
    w = np.maximum(res.x, 0)
    return w / w.sum()
def portfolio_stats(w, ret_df, instruments):
    pr = ret_df[instruments].values @ w
    cum = np.cumprod(1 + pr)
    n = len(pr)
    ann_ret = (cum[-1] ** (252 / n)) - 1
    ann_vol = np.std(pr) * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    mdd = ((cum / np.maximum.accumulate(cum)) - 1).min()
    return ann_ret, ann_vol, sharpe, mdd, cum, pr

In [4]:
TRAIN_END = '2024-02-28'
TEST_START = '2024-03-01'
TEST_END = '2024-05-31'
train_feat = feat_all[(feat_all.index <= pd.Timestamp(TRAIN_END))].dropna(subset=feat_cols + ['target'])
train_ret = returns.loc[:TRAIN_END]
test_ret = returns.loc[TEST_START:TEST_END]
print(f'train features: {len(train_feat)}')
print(f'train returns: {len(train_ret)} days')
print(f'test returns: {len(test_ret)} days ({len(test_ret)/21:.1f} months)')

train features: 22910
train returns: 2543 days
test returns: 92 days (4.4 months)


In [5]:
xgb_models = {}
rf_models = {}
xgb_preds = {}
rf_preds = {}

for inst in instruments:
    tr = train_feat[train_feat['instrument'] == inst]
    X_tr = tr[feat_cols].values
    y_tr = tr['target'].values
    
    xgb_m = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.5, reg_lambda=2.0,
        min_child_weight=10, random_state=42, verbosity=0)
    xgb_m.fit(X_tr, y_tr)
    xgb_models[inst] = xgb_m
    
    rf_m = RandomForestRegressor(
        n_estimators=500, max_depth=5, min_samples_leaf=30,
        max_features=0.5, random_state=42, n_jobs=-1)
    rf_m.fit(X_tr, y_tr)
    rf_models[inst] = rf_m
    
    inst_mask = (feat_all['instrument'] == inst) & (feat_all.index <= pd.Timestamp(TRAIN_END))
    last_row = feat_all[inst_mask][feat_cols].dropna().iloc[-1:]
    xgb_preds[inst] = xgb_m.predict(last_row.values)[0]
    rf_preds[inst] = rf_m.predict(last_row.values)[0]

ensemble_preds = {inst: 0.5 * xgb_preds[inst] + 0.5 * rf_preds[inst] for inst in instruments}

print(f'{"instrument":>15s} {"xgb":>10s} {"rf":>10s} {"ensemble":>10s}')
for inst in instruments:
    print(f'{inst:>15s} {xgb_preds[inst]:>10.6f} {rf_preds[inst]:>10.6f} {ensemble_preds[inst]:>10.6f}')

     instrument        xgb         rf   ensemble
   INSTRUMENT_1   0.000381   0.000469   0.000425
   INSTRUMENT_2   0.000545   0.000327   0.000436
   INSTRUMENT_3   0.000147   0.000236   0.000192
   INSTRUMENT_4   0.000058   0.000361   0.000210
   INSTRUMENT_5  -0.000036  -0.000329  -0.000182
   INSTRUMENT_6  -0.000018  -0.000157  -0.000088
   INSTRUMENT_7   0.000258   0.000903   0.000580
   INSTRUMENT_8   0.000206   0.000674   0.000440
   INSTRUMENT_9   0.002400   0.003574   0.002987
  INSTRUMENT_10   0.001780   0.001391   0.001585


In [6]:
mu_ens = np.array([ensemble_preds[i] for i in instruments]) * 252
cov_train = LedoitWolf().fit(train_ret[instruments].values).covariance_ * 252
w_val = optimize_mvo(mu_ens, cov_train, max_weight=0.40)

ann_ret, ann_vol, sharpe, mdd, cum, pr = portfolio_stats(w_val, test_ret, instruments)
print(f'validation')
print(f'return: {ann_ret:+.2%}, ann vol: {ann_vol:.2%}, sharpe: {sharpe:+.3f}, max dd: {mdd:+.2%}')
for inst, w in zip(instruments, w_val):
    print(f'{inst}: {w:.4f}')

validation
return: +25.61%, ann vol: 11.74%, sharpe: +2.182, max dd: -5.01%
INSTRUMENT_1: 0.2660
INSTRUMENT_2: 0.0000
INSTRUMENT_3: 0.0000
INSTRUMENT_4: 0.0000
INSTRUMENT_5: 0.0000
INSTRUMENT_6: 0.1336
INSTRUMENT_7: 0.4000
INSTRUMENT_8: 0.0422
INSTRUMENT_9: 0.1582
INSTRUMENT_10: 0.0000


In [7]:
wf_window = 504
wf_step = 63
wf_results = []

for end_idx in range(wf_window + 252, len(returns) - wf_step, wf_step):
    wf_train_start = max(0, end_idx - wf_window)
    wf_train_end = end_idx
    wf_test_start = end_idx
    wf_test_end = min(end_idx + wf_step, len(returns))    
    wf_train_dates = returns.index[wf_train_start:wf_train_end]
    wf_test_dates = returns.index[wf_test_start:wf_test_end]
    wf_xgb_preds = {}
    wf_rf_preds = {}
    for inst in instruments:
        inst_feat = feat_all[(feat_all['instrument'] == inst) & (feat_all.index.isin(wf_train_dates))]
        inst_feat = inst_feat.dropna(subset=feat_cols + ['target'])
        if len(inst_feat) < 50:
            wf_xgb_preds[inst] = 0.0
            wf_rf_preds[inst] = 0.0
            continue
        X_wf = inst_feat[feat_cols].values
        y_wf = inst_feat['target'].values
        xm = xgb.XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=2.0,
            min_child_weight=10, random_state=42, verbosity=0
        )
        xm.fit(X_wf, y_wf)
        rm = RandomForestRegressor(
            n_estimators=500, max_depth=5, min_samples_leaf=30,
            max_features=0.5, random_state=42, n_jobs=-1
        )
        rm.fit(X_wf, y_wf)
        last = inst_feat[feat_cols].iloc[-1:]
        wf_xgb_preds[inst] = xm.predict(last.values)[0]
        wf_rf_preds[inst] = rm.predict(last.values)[0]
    
    wf_ens = {inst: 0.5 * wf_xgb_preds[inst] + 0.5 * wf_rf_preds[inst] for inst in instruments}
    wf_mu = np.array([wf_ens[i] for i in instruments]) * 252
    wf_ret_train = returns[instruments].iloc[wf_train_start:wf_train_end]
    wf_cov = LedoitWolf().fit(wf_ret_train.values).covariance_ * 252
    wf_w = optimize_mvo(wf_mu, wf_cov, max_weight=0.40)
    wf_test_ret = returns[instruments].iloc[wf_test_start:wf_test_end]
    wf_pr = wf_test_ret.values @ wf_w
    wf_sharpe = wf_pr.mean() / wf_pr.std() * np.sqrt(252) if wf_pr.std() > 0 else 0
    wf_ann_ret = (np.prod(1 + wf_pr) ** (252 / len(wf_pr))) - 1
    wf_results.append({
        'period_start': wf_test_dates[0],
        'period_end': wf_test_dates[-1],
        'sharpe': wf_sharpe,
        'ann_ret': wf_ann_ret,
        'weights': wf_w
    })
    print(f'{wf_test_dates[0].date()} to {wf_test_dates[-1].date()}: sharpe={wf_sharpe:+.3f} Ret={wf_ann_ret:+.1%}')

wf_sharpes = [r['sharpe'] for r in wf_results]
print(f'mean sharpe: {np.mean(wf_sharpes):.3f}', 'std sharpe', np.std(wf_sharpes), 'min sharpe', np.min(wf_sharpes), 'max sharpe', np.max(wf_sharpes), '%poistive', np.mean(np.array(wf_sharpes) > 0))

2019-04-09 to 2019-06-10: sharpe=+4.491 Ret=+29.1%
2019-06-11 to 2019-08-12: sharpe=+4.675 Ret=+45.2%
2019-08-13 to 2019-10-14: sharpe=+0.359 Ret=+2.1%
2019-10-15 to 2019-12-16: sharpe=+2.978 Ret=+10.9%
2019-12-17 to 2020-02-17: sharpe=+7.971 Ret=+31.8%
2020-02-18 to 2020-04-20: sharpe=+2.397 Ret=+48.6%
2020-04-21 to 2020-06-22: sharpe=+3.250 Ret=+40.8%
2020-06-23 to 2020-08-24: sharpe=+4.044 Ret=+46.9%
2020-08-25 to 2020-10-26: sharpe=-0.313 Ret=-4.8%
2020-10-27 to 2020-12-28: sharpe=+5.036 Ret=+102.2%
2020-12-29 to 2021-03-01: sharpe=+3.623 Ret=+803.4%
2021-03-02 to 2021-05-03: sharpe=+4.208 Ret=+329.5%
2021-05-04 to 2021-07-05: sharpe=-0.464 Ret=-17.4%
2021-07-06 to 2021-09-06: sharpe=+4.229 Ret=+90.1%
2021-09-07 to 2021-11-08: sharpe=+2.383 Ret=+91.8%
2021-11-09 to 2022-01-10: sharpe=-1.354 Ret=-23.9%
2022-01-11 to 2022-03-14: sharpe=+2.856 Ret=+71.3%
2022-03-15 to 2022-05-16: sharpe=+0.562 Ret=+10.7%
2022-05-17 to 2022-07-18: sharpe=-1.341 Ret=-20.1%
2022-07-19 to 2022-09-19: shar

In [8]:
# full backtest
all_dates = []
all_returns = []
for r in wf_results:
    period_ret = returns[instruments].loc[r['period_start']:r['period_end']]
    pr = period_ret.values @ r['weights']
    all_dates.extend(period_ret.index.tolist())
    all_returns.extend(pr.tolist())

bt = pd.Series(all_returns, index=all_dates)
bt = bt[~bt.index.duplicated(keep='first')]
bt_cum = (1 + bt).cumprod()
total_days = len(bt)
total_ret = bt_cum.iloc[-1] - 1
ann_ret_bt = (1 + total_ret) ** (252 / total_days) - 1
ann_vol_bt = bt.std() * np.sqrt(252)
sharpe_bt = ann_ret_bt / ann_vol_bt
mdd_bt = ((bt_cum / bt_cum.cummax()) - 1).min()
print(f"period: {bt.index[0].date()} to {bt.index[-1].date()} ({total_days} days), total return {total_ret:+.2%}, annualized return: {ann_ret_bt:+.2%}, annualized vol: {ann_vol_bt:.2%}, sharpe ratio: {sharpe_bt:+.3f}, max drawdown: {mdd_bt:+.2%}")
ew_ret = returns[instruments].loc[bt.index[0]:bt.index[-1]].mean(axis=1)
ew_cum = (1 + ew_ret).cumprod()
ew_ann_ret = (ew_cum.iloc[-1] ** (252 / len(ew_ret))) - 1
ew_ann_vol = ew_ret.std() * np.sqrt(252)
ew_sharpe = ew_ann_ret / ew_ann_vol
print(f'{ew_sharpe:+.3f} ret={ew_ann_ret:+.2%}')

period: 2019-04-09 to 2024-12-16 (2079 days), total return +1891.70%, annualized return: +43.71%, annualized vol: 20.79%, sharpe ratio: +2.103, max drawdown: -21.68%
+0.919 ret=+15.42%


In [9]:
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ax = axes[0, 0]
# ax.plot(bt_cum.index, bt_cum.values, label='Ensemble', linewidth=2)
# ew_aligned = ew_cum.reindex(bt_cum.index).dropna()
# ax.plot(ew_aligned.index, ew_aligned.values, label='Equal Weight', linewidth=1, alpha=0.7)
# ax.set_title('Walk-Forward Cumulative Returns')
# ax.legend()
# ax.grid(True, alpha=0.3)

# ax = axes[0, 1]
# dd = (bt_cum / bt_cum.cummax()) - 1
# ax.fill_between(dd.index, dd.values, 0, alpha=0.5, color='red')
# ax.set_title('Drawdown')
# ax.grid(True, alpha=0.3)

# ax = axes[1, 0]
# ax.bar(range(len(wf_sharpes)), wf_sharpes, color=['green' if s > 0 else 'red' for s in wf_sharpes])
# ax.axhline(0, color='black', linewidth=0.5)
# ax.set_title('Walk-Forward Period Sharpe Ratios')
# ax.grid(True, alpha=0.3)

# ax = axes[1, 1]
# rolling_sharpe = bt.rolling(63).mean() / bt.rolling(63).std() * np.sqrt(252)
# ax.plot(rolling_sharpe.index, rolling_sharpe.values)
# ax.axhline(0, color='black', linewidth=0.5)
# ax.set_title('Rolling 63-day Sharpe')
# ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

In [10]:
FINAL_END = '2025-02-28'
final_feat = feat_all[(feat_all.index <= pd.Timestamp(FINAL_END))].dropna(subset=feat_cols + ['target'])
final_xgb_preds = {}
final_rf_preds = {}

for inst in instruments:
    tr = final_feat[final_feat['instrument'] == inst]
    X_tr = tr[feat_cols].values
    y_tr = tr['target'].values
    
    xgb_m = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.5, reg_lambda=2.0,
        min_child_weight=10, random_state=42, verbosity=0
    )
    xgb_m.fit(X_tr, y_tr)
    
    rf_m = RandomForestRegressor(
        n_estimators=500, max_depth=5, min_samples_leaf=30,
        max_features=0.5, random_state=42, n_jobs=-1
    )
    rf_m.fit(X_tr, y_tr)
    
    inst_mask = (feat_all['instrument'] == inst) & (feat_all.index <= pd.Timestamp(FINAL_END))
    last_row = feat_all[inst_mask][feat_cols].dropna().iloc[-1:]
    final_xgb_preds[inst] = xgb_m.predict(last_row.values)[0]
    final_rf_preds[inst] = rf_m.predict(last_row.values)[0]

final_ens_preds = {inst: 0.5 * final_xgb_preds[inst] + 0.5 * final_rf_preds[inst] for inst in instruments}
for inst in instruments:
    print(f'{inst:>15s} {final_xgb_preds[inst]:>10.6f} {final_rf_preds[inst]:>10.6f} {final_ens_preds[inst]:>10.6f}')

   INSTRUMENT_1   0.000409   0.000558   0.000483
   INSTRUMENT_2   0.000562   0.000543   0.000553
   INSTRUMENT_3   0.000133   0.001155   0.000644
   INSTRUMENT_4   0.000085   0.000328   0.000207
   INSTRUMENT_5  -0.000024   0.000545   0.000260
   INSTRUMENT_6  -0.000014   0.000249   0.000117
   INSTRUMENT_7   0.000301   0.001026   0.000664
   INSTRUMENT_8  -0.000572  -0.000743  -0.000657
   INSTRUMENT_9   0.002070   0.001657   0.001864
  INSTRUMENT_10   0.000859   0.002045   0.001452


In [11]:
mu_final = np.array([final_ens_preds[i] for i in instruments]) * 252
cov_final = LedoitWolf().fit(returns[instruments].values).covariance_ * 252

w_final = optimize_mvo(mu_final, cov_final, max_weight=0.40)

w_final = np.round(w_final, 4)
residual = round(1.0 - w_final.sum(), 4)
w_final[np.argmax(w_final)] += residual

assert abs(w_final.sum() - 1.0) < 1e-8
assert all(w_final >= 0)
for inst, w in zip(instruments, w_final):
    print(f'  {inst}: {w:.4f}')
print(f'sum: {w_final.sum():.4f}')

  INSTRUMENT_1: 0.0000
  INSTRUMENT_2: 0.0000
  INSTRUMENT_3: 0.3288
  INSTRUMENT_4: 0.0000
  INSTRUMENT_5: 0.1413
  INSTRUMENT_6: 0.0825
  INSTRUMENT_7: 0.4001
  INSTRUMENT_8: 0.0000
  INSTRUMENT_9: 0.0473
  INSTRUMENT_10: 0.0000
sum: 1.0000


In [12]:
alloc = pd.DataFrame({'asset': instruments, 'weight': w_final})
alloc.to_csv('/Users/yash/algothon-26/labubu-money/allocation.csv', index=False)
print(alloc.to_string(index=False))
print(f'\nsaved to allocation.csv')

        asset  weight
 INSTRUMENT_1  0.0000
 INSTRUMENT_2  0.0000
 INSTRUMENT_3  0.3288
 INSTRUMENT_4  0.0000
 INSTRUMENT_5  0.1413
 INSTRUMENT_6  0.0825
 INSTRUMENT_7  0.4001
 INSTRUMENT_8  0.0000
 INSTRUMENT_9  0.0473
INSTRUMENT_10  0.0000

saved to allocation.csv


In [13]:
VAL_PATH = '/Users/yash/algothon-26/man-imperial-algothon-2026/data/2025-02-28'
val_prices = pd.read_csv(os.path.join(VAL_PATH, 'prices.csv'), parse_dates=['date']).sort_values('date').set_index('date')
val_returns = val_prices[instruments].pct_change().dropna()
oos_returns = val_returns.loc['2025-01-01':'2025-02-28']

print(f'training period: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'OOS period:{oos_returns.index[0].date()} to {oos_returns.index[-1].date()} ({len(oos_returns)} days)')
for inst, w in zip(instruments, w_final):
    print(f'{inst}: {w:.4f}')

ann_ret_oos, ann_vol_oos, sharpe_oos, mdd_oos, cum_oos, pr_oos = portfolio_stats(w_final, oos_returns, instruments)

print(f'oos perf. ann_ret {ann_ret_oos:+.2%}, ann_vol {ann_vol_oos:.2%}, sharpe {sharpe_oos:+.3f}, max dd {mdd_oos:+.2%}, cum ret {cum_oos[-1]-1:+.2%}')
print(f'\nper-instrument OOS cumulative returns:')
for inst in instruments:
    inst_cum = (1 + oos_returns[inst]).prod() - 1
    print(f'{inst}: {inst_cum:+.2%}  (weight: {w_final[instruments.index(inst)]:.4f})')

# Equal weight benchmark
ew_pr_oos = oos_returns[instruments].mean(axis=1)
ew_cum = (1 + ew_pr_oos).cumprod()
ew_sr = ew_pr_oos.mean() / ew_pr_oos.std() * np.sqrt(252) if ew_pr_oos.std() > 0 else 0
print(f'\new bench={ew_sr:+.3f}, cum Return={ew_cum.iloc[-1]-1:+.2%}')

training period: 2017-01-04 to 2024-12-31
OOS period:2025-01-01 to 2025-02-28 (59 days)
INSTRUMENT_1: 0.0000
INSTRUMENT_2: 0.0000
INSTRUMENT_3: 0.3288
INSTRUMENT_4: 0.0000
INSTRUMENT_5: 0.1413
INSTRUMENT_6: 0.0825
INSTRUMENT_7: 0.4001
INSTRUMENT_8: 0.0000
INSTRUMENT_9: 0.0473
INSTRUMENT_10: 0.0000
oos perf. ann_ret +32.40%, ann_vol 7.07%, sharpe +4.581, max dd -1.25%, cum ret +6.79%

per-instrument OOS cumulative returns:
INSTRUMENT_1: +1.35%  (weight: 0.0000)
INSTRUMENT_2: -0.61%  (weight: 0.0000)
INSTRUMENT_3: +7.83%  (weight: 0.3288)
INSTRUMENT_4: +3.33%  (weight: 0.0000)
INSTRUMENT_5: +6.19%  (weight: 0.1413)
INSTRUMENT_6: +3.10%  (weight: 0.0825)
INSTRUMENT_7: +8.77%  (weight: 0.4001)
INSTRUMENT_8: -0.40%  (weight: 0.0000)
INSTRUMENT_9: -9.92%  (weight: 0.0473)
INSTRUMENT_10: -33.28%  (weight: 0.0000)

ew bench=-0.594, cum Return=-1.69%


In [14]:
ALL_PATH = '/Users/yash/algothon-26/man-imperial-algothon-2026/data/2025-02-28'
prices_all = pd.read_csv(os.path.join(ALL_PATH, 'prices.csv'), parse_dates=['date']).sort_values('date').set_index('date')
signals_all = pd.read_csv(os.path.join(ALL_PATH, 'signals.csv'), parse_dates=['date']).sort_values('date').set_index('date')
volumes_all = pd.read_csv(os.path.join(ALL_PATH, 'volumes.csv'), parse_dates=['date']).sort_values('date').set_index('date')
cash_all = pd.read_csv(os.path.join(ALL_PATH, 'cash_rate.csv'), parse_dates=['date']).sort_values('date').set_index('date')
returns_all = prices_all[instruments].pct_change().dropna()
print(f'full dataset: {returns_all.index[0].date()} to {returns_all.index[-1].date()} ({len(returns_all)} days)')

feat_full, feat_cols_full = build_features(returns_all, signals_all, volumes_all, cash_all, instruments, FORWARD)
train_full = feat_full.dropna(subset=feat_cols_full + ['target'])
print(f'train samples: {len(train_full)}')
final2_xgb_preds = {}
final2_rf_preds = {}
for inst in instruments:
    tr = train_full[train_full['instrument'] == inst]
    X_tr = tr[feat_cols_full].values
    y_tr = tr['target'].values
    
    xgb_m = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.5, reg_lambda=2.0,
        min_child_weight=10, random_state=42, verbosity=0
    )
    xgb_m.fit(X_tr, y_tr)
    
    rf_m = RandomForestRegressor(
        n_estimators=500, max_depth=5, min_samples_leaf=30,
        max_features=0.5, random_state=42, n_jobs=-1
    )
    rf_m.fit(X_tr, y_tr)
    last_row = feat_full[(feat_full['instrument'] == inst)][feat_cols_full].dropna().iloc[-1:]
    final2_xgb_preds[inst] = xgb_m.predict(last_row.values)[0]
    final2_rf_preds[inst] = rf_m.predict(last_row.values)[0]

final2_ens = {inst: 0.5 * final2_xgb_preds[inst] + 0.5 * final2_rf_preds[inst] for inst in instruments}

print(f'\n{"instrument":>15s} {"xgb":>10s} {"rf":>10s} {"ens":>10s}')
for inst in instruments:
    print(f'{inst:>15s} {final2_xgb_preds[inst]:>10.6f} {final2_rf_preds[inst]:>10.6f} {final2_ens[inst]:>10.6f}')

mu_new = np.array([final2_ens[i] for i in instruments]) * 252
cov_new = LedoitWolf().fit(returns_all[instruments].values).covariance_ * 252
w_new = optimize_mvo(mu_new, cov_new, max_weight=0.40)
w_new = np.round(w_new, 4)
residual = round(1.0 - w_new.sum(), 4)
w_new[np.argmax(w_new)] += residual

for inst, w in zip(instruments, w_new):
    print(f'  {inst}: {w:.4f}')
print(f'sum: {w_new.sum():.4f}')

full dataset: 2017-01-04 to 2025-02-28 (2909 days)
train samples: 25940

     instrument        xgb         rf        ens
   INSTRUMENT_1   0.000403   0.000489   0.000446
   INSTRUMENT_2   0.000558   0.000513   0.000536
   INSTRUMENT_3   0.000134   0.000040   0.000087
   INSTRUMENT_4   0.000084   0.000453   0.000268
   INSTRUMENT_5  -0.000033   0.000077   0.000022
   INSTRUMENT_6  -0.000018  -0.000174  -0.000096
   INSTRUMENT_7   0.000313   0.000397   0.000355
   INSTRUMENT_8  -0.000531  -0.000767  -0.000649
   INSTRUMENT_9   0.001512   0.001214   0.001363
  INSTRUMENT_10  -0.000073   0.001069   0.000498
  INSTRUMENT_1: 0.4000
  INSTRUMENT_2: 0.0357
  INSTRUMENT_3: 0.0000
  INSTRUMENT_4: 0.0000
  INSTRUMENT_5: 0.0916
  INSTRUMENT_6: 0.0000
  INSTRUMENT_7: 0.4000
  INSTRUMENT_8: 0.0000
  INSTRUMENT_9: 0.0727
  INSTRUMENT_10: 0.0000
sum: 1.0000


In [15]:
alloc_new = pd.DataFrame({'asset': instruments, 'weight': w_new})
alloc_new.to_csv('/Users/yash/algothon-26/labubu-money/allocation.csv', index=False)
print(alloc_new.to_string(index=False))
print(f'\nSaved to /Users/yash/algothon-26/labubu-money/allocation.csv')


        asset  weight
 INSTRUMENT_1  0.4000
 INSTRUMENT_2  0.0357
 INSTRUMENT_3  0.0000
 INSTRUMENT_4  0.0000
 INSTRUMENT_5  0.0916
 INSTRUMENT_6  0.0000
 INSTRUMENT_7  0.4000
 INSTRUMENT_8  0.0000
 INSTRUMENT_9  0.0727
INSTRUMENT_10  0.0000

Saved to /Users/yash/algothon-26/labubu-money/allocation.csv
